# Surface Energy

Calculate the surface energy of a slab using a DFT workflow on the Mat3ra platform.

This notebook is designed around the updated Surface Energy workflow that resolves the bulk material in this order:

- `metadata.bulkId`
- `metadata.build[*].configuration.stack_components[0].crystal._id`
- `...scaledHash`
- `...hash`
- recomputed hash as a last fallback

Before running the Surface Energy workflow, the notebook checks whether the exact bulk representation found in slab metadata exists on the platform and whether a refined bulk total-energy property is already available. If the refined bulk total energy is missing, the notebook automatically runs a bulk Total Energy workflow first and then proceeds to the Surface Energy workflow.

<h2 style="color:green">Usage</h2>

1. Put a slab material JSON into `../uploads`, or set `SLAB_NAME` to match an existing slab on the platform.
1. Set the slab and calculation parameters in cells 1.2 and 1.3 below.
1. Click "Run" > "Run All" to run all cells.
1. Wait for the bulk Total Energy job, if needed, and the Surface Energy job to complete.
1. Scroll down to inspect the resolved bulk, the bulk total energy, and the final surface energy.

## Summary

1. Set up the environment and parameters: install packages (JupyterLite only) and configure slab, workflow, and compute parameters.
1. Authenticate and initialize API client: authenticate via browser, initialize the client, then select account and project.
1. Load slab and resolve the exact bulk candidate from slab metadata: inspect `metadata.build[*].configuration.stack_components[0].crystal`, resolve the first usable bulk identity, and ensure the exact bulk material exists on the platform.
1. Ensure bulk total energy exists: query refined bulk total energy for the resolved exact bulk and run a bulk Total Energy workflow only if the property is missing.
1. Configure Surface Energy workflow: load the standard Surface Energy workflow from Standata, apply QE model and method settings, and save the workflow.
1. Configure compute: get list of clusters and create compute configuration with selected cluster, queue, and number of processors.
1. Create the Surface Energy job with slab and workflow configuration: update slab `metadata.bulkId` to the exact bulk used in this notebook and create the job.
1. Submit the job and monitor the status: submit the job and wait for completion.
1. Retrieve results: get and display the surface energy together with the slab total energy and the resolved bulk total energy.


## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)


In [ ]:
import sys

if sys.platform == "emscripten":
    import micropip

    await micropip.install("mat3ra-api-examples", deps=False)
    await micropip.install("mat3ra-utils")
    from mat3ra.utils.jupyterlite.packages import install_packages

    await install_packages("api_examples")


### 1.2. Set parameters and configurations for the workflow and job


In [ ]:
from datetime import datetime
from mat3ra.ide.compute import QueueName

# 2. Auth and organization parameters
# Set organization name to use it as the owner, otherwise your personal account is used
ORGANIZATION_NAME = None

# 3. Material parameters
FOLDER = "../uploads"
SLAB_NAME = "Slab"  # Name of the slab to load from uploads folder or from the platform

# 4. Workflow parameters
SURFACE_WORKFLOW_SEARCH_TERM = "surface_energy.json"  # Search term for Workflows Standata
TOTAL_ENERGY_WORKFLOW_SEARCH_TERM = "total_energy.json"  # Search term for Workflows Standata
APPLICATION_NAME = "espresso"  # Specify application name (e.g., "espresso", "vasp", "nwchem")
ADD_RELAXATION = None  # Whether to add relaxation subworkflow as first unit for bulk total energy

# set k-grid for relaxation and SCF (if not set, KPPRA is used by default)
RELAXATION_KGRID = None  # e.g., [4, 4, 4]
SCF_KGRID = None  # e.g., [4, 4, 1]

MY_SURFACE_WORKFLOW_NAME = "Surface Energy"

# 5. Compute parameters
CLUSTER_NAME = None  # specify full or partial name i.e. "cluster-001" to select
QUEUE_NAME = QueueName.D
PPN = 1

# 6. Job parameters
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 30  # seconds


## 2. Authenticate and initialize API client
### 2.1. Authenticate
Authenticate in the browser and have credentials stored in environment variable `OIDC_ACCESS_TOKEN`.


In [ ]:
from utils.auth import authenticate

await authenticate()


### 2.2. Initialize API client


In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client


### 2.3. Select account to work under


In [ ]:
client.list_accounts()


In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"✅ Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")


### 2.4. Select project


In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"✅ Using project: {projects[0]['name']} ({project_id})")


## 3. Load slab and resolve the exact bulk candidate
### 3.1. Load slab from local file or platform


In [ ]:
from mat3ra.made.material import Material
from utils.jupyterlite import load_material_from_folder
from utils.visualize import visualize_materials as visualize

slab = load_material_from_folder(FOLDER, SLAB_NAME)

if slab is None:
    slab_matches = client.materials.list({
        "name": {"$regex": SLAB_NAME, "$options": "i"},
        "owner._id": ACCOUNT_ID,
    })
    if not slab_matches:
        raise ValueError(
            f"No slab containing '{SLAB_NAME}' was found in '{FOLDER}' or in account '{ACCOUNT_ID}'."
        )
    slab = Material.create(slab_matches[0])
    print(f"♻️  Loaded slab from platform: {slab_matches[0]['_id']}")
else:
    print(f"✅ Loaded slab from folder: {slab.name}")

visualize(slab)


### 3.2. Inspect slab metadata and check that bulk will be found


In [ ]:
import copy
import json
from typing import Optional

from utils.visualize import display_JSON


CRYSTAL_CREATE_KEYS = {
    "name",
    "basis",
    "lattice",
    "formula",
    "unitCellFormula",
    "metadata",
    "derivedProperties",
    "isNonPeriodic",
    "schemaVersion",
}


def material_from_crystal(crystal: dict) -> Material:
    payload = {key: copy.deepcopy(value) for key, value in crystal.items() if key in CRYSTAL_CREATE_KEYS}
    return Material.create(payload)


def get_primary_bulk_crystal(slab_material: Material) -> Optional[dict]:
    slab_dict = slab_material.to_dict()
    metadata = slab_dict.get("metadata") or {}
    for build_step in reversed(metadata.get("build") or []):
        try:
            return build_step["configuration"]["stack_components"][0]["crystal"]
        except (KeyError, IndexError, TypeError):
            continue
    return None


def resolve_bulk_reference(crystal: dict) -> Optional[dict]:
    if crystal.get("_id") is not None:
        return {"kind": "id", "value": crystal["_id"]}
    if crystal.get("scaledHash") is not None:
        return {"kind": "scaledHash", "value": crystal["scaledHash"]}
    if crystal.get("hash") is not None:
        return {"kind": "hash", "value": crystal["hash"]}
    try:
        return {"kind": "hash", "value": material_from_crystal(crystal).hash}
    except Exception:
        return None


bulk_crystal = get_primary_bulk_crystal(slab)
if bulk_crystal is None:
    raise ValueError(
        "No metadata.build[*].configuration.stack_components[0].crystal entry was found on the slab."
    )

bulk_resolution = resolve_bulk_reference(bulk_crystal)
print(f"Resolved bulk reference: {bulk_resolution}")
display_JSON({"bulk_resolution": bulk_resolution, "bulk_crystal": bulk_crystal}, level=3)


## 5. Configure the Surface Energy workflow
### 5.1. Load workflow from Standata and preview it


In [ ]:
surface_workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(
    SURFACE_WORKFLOW_SEARCH_TERM
)
surface_workflow = Workflow.create(surface_workflow_config)
surface_workflow.name = MY_SURFACE_WORKFLOW_NAME
surface_workflow = apply_workflow_kgrids(surface_workflow, scf_kgrid=SCF_KGRID)

visualize_workflow(surface_workflow)


### 5.2. Save workflow to collection


In [ ]:
saved_surface_workflow_response = get_or_create_workflow(client, surface_workflow, ACCOUNT_ID)
saved_surface_workflow = Workflow.create(saved_surface_workflow_response)
print(f"Surface workflow ID: {saved_surface_workflow.id}")


## 6. Create the compute configuration
### 6.1. Select cluster


In [ ]:
clusters = client.clusters.list()
print(f"Available clusters: {[c['hostname'] for c in clusters]}")


### 6.2. Create compute configuration


In [ ]:
from mat3ra.ide.compute import Compute

if CLUSTER_NAME:
    cluster = next((c for c in clusters if CLUSTER_NAME in c["hostname"]), None)
else:
    cluster = clusters[0]

compute = Compute(cluster=cluster, queue=QUEUE_NAME, ppn=PPN)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, ppn: {PPN}")


## 7. Create the Surface Energy job
### 7.1. Create job with slab and workflow configuration


In [ ]:
print(f"Slab material: {saved_slab.id}")
print(f"Resolved exact bulk material: {bulk_material.id}")
print(f"Surface workflow: {saved_surface_workflow.id}")
print(f"Project: {project_id}")

surface_job_name = f"{MY_SURFACE_WORKFLOW_NAME} {saved_slab.formula} {timestamp}"
surface_job_response = create_job(
    api_client=client,
    materials=[saved_slab],
    workflow=surface_workflow,
    project_id=project_id,
    owner_id=ACCOUNT_ID,
    prefix=surface_job_name,
    compute=compute.to_dict(),
)

surface_job = dict_to_namespace(surface_job_response)
surface_job_id = surface_job._id
print(f"✅ Surface Energy job created successfully: {surface_job_id}")
display_JSON(surface_job_response)


### 7.2. Submit the Surface Energy job and monitor the status


In [ ]:
client.jobs.submit(surface_job_id)
print(f"✅ Job {surface_job_id} submitted successfully!")

In [ ]:

await wait_for_jobs_to_finish_async(client.jobs, [surface_job_id], poll_interval=POLL_INTERVAL)


## 8. Retrieve results
### 8.1. Retrieve and visualize Surface Energy results


In [ ]:
from mat3ra.prode import PropertyName
from utils.visualize import visualize_properties

surface_energy_data = client.properties.get_for_job(surface_job_id, property_name="surface_energy")
slab_total_energy_data = client.properties.get_for_job(
    surface_job_id, property_name=PropertyName.scalar.total_energy.value
)

if surface_energy_data:
    visualize_properties(surface_energy_data, title="Surface Energy")
else:
    print("No 'surface_energy' property was returned for the job.")

if slab_total_energy_data:
    visualize_properties(slab_total_energy_data, title="Slab Total Energy")
else:
    print("No slab total energy property was returned for the job.")

print(f"Resolved bulk reference from slab metadata: {bulk_resolution}")
print(f"Exact bulk material used by the workflow: {bulk_material_response['_id']}")
print(f"Exact bulk exabyteId: {bulk_material_response['exabyteId']}")
print(f"Refined bulk total energy used by the workflow: {bulk_total_energy_property['data']['value']}")
print(f"Saved slab bulkId used by the workflow: {(saved_slab_response.get('metadata') or {}).get('bulkId')}")
